In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


In [12]:
df = pd.read_csv('./data/flood.csv')


In [13]:

# Define features and target
X = df.drop('FloodProbability', axis=1)
y = df['FloodProbability']

print("🔍 COMPREHENSIVE DATA LEAKAGE DETECTION")
print("=" * 50)

# 1. Basic Data Quality Checks
print("=== 1. BASIC DATA QUALITY ===")
print(f"Dataset shape: {df.shape}")
print(f"Features: {len(X.columns)}")
print(f"Target variable: FloodProbability")

# Check for duplicates
duplicate_rows = df.duplicated().sum()
duplicate_features = df.drop('FloodProbability', axis=1).duplicated().sum()
print(f"Duplicate rows (entire): {duplicate_rows}")
print(f"Duplicate feature combinations: {duplicate_features}")

# Target distribution analysis
print(f"\n=== 2. TARGET VARIABLE ANALYSIS ===")
print(f"Unique target values: {y.nunique()}")
print(f"Target range: {y.min():.6f} to {y.max():.6f}")
print(f"Target std deviation: {y.std():.6f}")
print(f"Target variance: {y.var():.6f}")

# Check if target is too uniform or has suspicious patterns
if y.std() < 0.01:
    print("⚠️  WARNING: Target has very low variance - potential data issue")
if y.nunique() < 10:
    print("⚠️  WARNING: Target has very few unique values")

🔍 COMPREHENSIVE DATA LEAKAGE DETECTION
=== 1. BASIC DATA QUALITY ===
Dataset shape: (50000, 21)
Features: 20
Target variable: FloodProbability
Duplicate rows (entire): 0
Duplicate feature combinations: 0

=== 2. TARGET VARIABLE ANALYSIS ===
Unique target values: 83
Target range: 0.285000 to 0.725000
Target std deviation: 0.050034
Target variance: 0.002503


In [14]:

# Compute correlation matrix
corr_matrix = df.corr()

print("\n=== 3. PERFECT CORRELATION DETECTION ===")

# Calculate correlations with target
correlations = corr_matrix['FloodProbability'].abs().sort_values(ascending=False)
print("All feature correlations with target:")
print(correlations)

# Flag suspicious correlations
high_corr_features = correlations[correlations > 0.95]
if len(high_corr_features) > 1:  # Excluding target itself
    print(f"\n🚨 CRITICAL: {len(high_corr_features)-1} features with correlation > 0.95:")
    for feature, corr in high_corr_features.items():
        if feature != 'FloodProbability':
            print(f"  {feature}: {corr:.6f}")

# Check for multicollinearity between features
print(f"\n=== 4. FEATURE-TO-FEATURE CORRELATIONS ===")
feature_corr = corr_matrix.drop('FloodProbability', axis=1).drop('FloodProbability', axis=0)
high_feature_corr = np.where(np.abs(feature_corr) > 0.95)
high_corr_pairs = [(feature_corr.index[i], feature_corr.columns[j]) 
                   for i, j in zip(high_feature_corr[0], high_feature_corr[1]) if i != j]

if high_corr_pairs:
    print("🚨 High correlation pairs between features:")
    for pair in high_corr_pairs[:10]:  # Show first 10
        corr_val = feature_corr.loc[pair[0], pair[1]]
        print(f"  {pair[0]} ↔ {pair[1]}: {corr_val:.6f}")
else:
    print("✅ No suspicious feature-to-feature correlations found")


=== 3. PERFECT CORRELATION DETECTION ===
All feature correlations with target:
FloodProbability                   1.000000
DeterioratingInfrastructure        0.229444
TopographyDrainage                 0.229414
RiverManagement                    0.228917
Watersheds                         0.228152
DamsQuality                        0.227467
PopulationScore                    0.226928
Siltation                          0.226544
IneffectiveDisasterPreparedness    0.225126
PoliticalFactors                   0.225009
MonsoonIntensity                   0.224081
WetlandLoss                        0.223732
InadequatePlanning                 0.223329
Landslides                         0.222991
AgriculturalPractices              0.221846
ClimateChange                      0.220986
Urbanization                       0.220867
Deforestation                      0.220237
Encroachments                      0.218259
DrainageSystems                    0.217895
CoastalVulnerability               0.215

In [15]:
# Test if any single feature can predict the target perfectly
print("\n=== 5. SINGLE FEATURE LEAKAGE TEST ===")
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

leakage_candidates = []

for feature in X.columns:
    # Test single feature prediction
    single_feature = X[[feature]].values.reshape(-1, 1)
    lr_single = LinearRegression()
    lr_single.fit(single_feature, y)
    single_r2 = lr_single.score(single_feature, y)
    
    if single_r2 > 0.9:
        leakage_candidates.append((feature, single_r2))
        print(f"🚨 LEAKAGE DETECTED: {feature} R² = {single_r2:.6f}")

if not leakage_candidates:
    print("✅ No single-feature leakage detected")
else:
    print(f"\n⚠️  Found {len(leakage_candidates)} potential leakage features")


=== 5. SINGLE FEATURE LEAKAGE TEST ===
✅ No single-feature leakage detected


In [16]:
# Check if target can be perfectly reconstructed from features
print("\n=== 6. LINEAR COMBINATION LEAKAGE TEST ===")

# Fit a simple linear regression to all features
lr_all = LinearRegression()
lr_all.fit(X, y)
y_pred_linear = lr_all.predict(X)
linear_r2 = r2_score(y, y_pred_linear)

print(f"Linear regression R² on all features: {linear_r2:.6f}")

if linear_r2 > 0.99:
    print("🚨 CRITICAL: Linear combination can perfectly predict target!")
    
    # Find the most important coefficients
    feature_coeffs = pd.DataFrame({
        'feature': X.columns,
        'coefficient': np.abs(lr_all.coef_)
    }).sort_values('coefficient', ascending=False)
    
    print("Top linear coefficients (potential leakage sources):")
    print(feature_coeffs.head(10))
elif linear_r2 > 0.95:
    print("⚠️  WARNING: Very high linear predictability - possible leakage")
else:
    print("✅ Linear combination test passed")


=== 6. LINEAR COMBINATION LEAKAGE TEST ===
Linear regression R² on all features: 1.000000
🚨 CRITICAL: Linear combination can perfectly predict target!
Top linear coefficients (potential leakage sources):
                            feature  coefficient
0                  MonsoonIntensity        0.005
5                     ClimateChange        0.005
7                         Siltation        0.005
15      DeterioratingInfrastructure        0.005
17                      WetlandLoss        0.005
3                     Deforestation        0.005
10  IneffectiveDisasterPreparedness        0.005
1                TopographyDrainage        0.005
2                   RiverManagement        0.005
18               InadequatePlanning        0.005


In [17]:
# Check your engineered features for leakage
print("\n=== 7. ENGINEERED FEATURES LEAKAGE CHECK ===")

engineered_features = [
    'Monsoon_Drainage_Interaction', 
    'Urbanization_Encroachment',
    'ClimateVulnerability', 
    'InfrastructureQuality'
]

print("Checking engineered features:")
for feature in engineered_features:
    if feature in correlations:
        corr_val = correlations[feature]
        print(f"  {feature}: {corr_val:.6f}")
        if corr_val > 0.95:
            print(f"    🚨 POTENTIAL LEAKAGE in engineered feature!")
        else:
            print(f"    ✅ No leakage detected in {feature}")


=== 7. ENGINEERED FEATURES LEAKAGE CHECK ===
Checking engineered features:


In [18]:
# Analyze if data might be artificially generated
print("\n=== 8. DATA GENERATION PATTERN ANALYSIS ===")

# Check for too-perfect distributions
feature_stats = []
for col in X.columns:
    stats = {
        'feature': col,
        'unique_values': X[col].nunique(),
        'min_val': X[col].min(),
        'max_val': X[col].max(),
        'std': X[col].std(),
        'range': X[col].max() - X[col].min()
    }
    feature_stats.append(stats)

stats_df = pd.DataFrame(feature_stats)
print("Feature statistics summary:")
print(stats_df.head(10))

# Check for suspicious integer-only values
integer_features = stats_df[stats_df['feature'].apply(lambda x: X[x].dtype in ['int64', 'int32'])]
print(f"\nInteger-only features: {len(integer_features)}")

# Check for limited value ranges (potential synthetic data indicator)
limited_range = stats_df[stats_df['range'] <= 20]
print(f"Features with range ≤ 20: {len(limited_range)} (potential synthetic data)")


=== 8. DATA GENERATION PATTERN ANALYSIS ===
Feature statistics summary:
                 feature  unique_values  min_val  max_val       std  range
0       MonsoonIntensity             17        0       16  2.236834     16
1     TopographyDrainage             18        0       18  2.246488     18
2        RiverManagement             17        0       16  2.231310     16
3          Deforestation             18        0       17  2.222743     17
4           Urbanization             18        0       17  2.243159     17
5          ClimateChange             18        0       17  2.226761     17
6            DamsQuality             17        0       16  2.245000     16
7              Siltation             17        0       16  2.232642     16
8  AgriculturalPractices             17        0       16  2.234588     16
9          Encroachments             18        0       18  2.241633     18

Integer-only features: 20
Features with range ≤ 20: 19 (potential synthetic data)


In [19]:
# Based on findings, implement remediation
print("\n=== 9. REMEDIATION RECOMMENDATIONS ===")

# Create a clean dataset
df_clean = df.copy()
X_clean = X.copy()
y_clean = y.copy()

removed_features = []

# Remove high-correlation features
if len(high_corr_features) > 1:
    for feature, corr in high_corr_features.items():
        if feature != 'FloodProbability' and corr > 0.98:
            if feature in X_clean.columns:
                X_clean = X_clean.drop(feature, axis=1)
                removed_features.append(f"{feature} (correlation: {corr:.6f})")
                print(f"🗑️  Removed {feature} - correlation too high: {corr:.6f}")

# Remove leakage candidates
for feature, r2 in leakage_candidates:
    if feature in X_clean.columns:
        X_clean = X_clean.drop(feature, axis=1)
        removed_features.append(f"{feature} (single feature R²: {r2:.6f})")
        print(f"🗑️  Removed {feature} - single feature leakage: {r2:.6f}")

print(f"\nTotal features removed: {len(removed_features)}")
print(f"Remaining features: {len(X_clean.columns)}")
print(f"Original features: {len(X.columns)}")


=== 9. REMEDIATION RECOMMENDATIONS ===

Total features removed: 0
Remaining features: 20
Original features: 20


In [20]:
# Test the cleaned dataset
print("\n=== 10. CLEAN DATASET VALIDATION ===")

if len(X_clean.columns) > 0:
    # Split clean data
    X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
        X_clean, y_clean, test_size=0.2, random_state=42
    )
    
    # Test with simple models on clean data
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.linear_model import LinearRegression
    
    models_clean = {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(n_estimators=50, random_state=42)
    }
    
    print("Testing models on CLEAN dataset:")
    for name, model in models_clean.items():
        model.fit(X_train_clean, y_train_clean)
        train_score = model.score(X_train_clean, y_train_clean)
        test_score = model.score(X_test_clean, y_test_clean)
        
        print(f"{name}:")
        print(f"  Train R²: {train_score:.4f}")
        print(f"  Test R²: {test_score:.4f}")
        print(f"  Difference: {train_score - test_score:.4f}")
        
        if test_score > 0.99:
            print(f"  🚨 Still suspicious - further investigation needed")
        elif test_score > 0.7:
            print(f"  ✅ Reasonable performance")
        else:
            print(f"  ⚠️  Low performance - may have removed too much")
else:
    print("❌ All features removed - dataset needs reconstruction")


=== 10. CLEAN DATASET VALIDATION ===
Testing models on CLEAN dataset:
Linear Regression:
  Train R²: 1.0000
  Test R²: 1.0000
  Difference: 0.0000
  🚨 Still suspicious - further investigation needed
Random Forest:
  Train R²: 0.9589
  Test R²: 0.7222
  Difference: 0.2366
  ✅ Reasonable performance


In [21]:
# Generate comprehensive report
print("\n" + "="*60)
print("🎯 DATA LEAKAGE INVESTIGATION SUMMARY")
print("="*60)

print(f"Original dataset: {df.shape[0]} rows, {len(X.columns)} features")
print(f"Cleaned dataset: {df.shape[0]} rows, {len(X_clean.columns)} features")
print(f"Features removed: {len(removed_features)}")

if removed_features:
    print("\nRemoved features:")
    for feature in removed_features:
        print(f"  - {feature}")

print(f"\nRecommendations:")
if len(X_clean.columns) == 0:
    print("❌ CRITICAL: All features removed - dataset likely synthetic")
    print("   → Recommendation: Find a different, real-world dataset")
elif len(removed_features) > len(X.columns) * 0.5:
    print("⚠️  WARNING: >50% features removed - dataset quality questionable")
    print("   → Recommendation: Verify data source and collection methodology")
elif linear_r2 > 0.99:
    print("🚨 CRITICAL: Perfect linear relationships detected")
    print("   → Recommendation: Dataset likely artificially generated")
else:
    print("✅ Dataset appears usable after cleaning")
    print("   → Recommendation: Proceed with cleaned dataset")

print(f"\nNext steps:")
print("1. If dataset is synthetic: Find real flood data from NOAA, USGS, etc.")
print("2. If dataset is questionable: Validate with domain experts")
print("3. If dataset is clean: Proceed with realistic performance expectations")


🎯 DATA LEAKAGE INVESTIGATION SUMMARY
Original dataset: 50000 rows, 20 features
Cleaned dataset: 50000 rows, 20 features
Features removed: 0

Recommendations:
🚨 CRITICAL: Perfect linear relationships detected
   → Recommendation: Dataset likely artificially generated

Next steps:
1. If dataset is synthetic: Find real flood data from NOAA, USGS, etc.
2. If dataset is questionable: Validate with domain experts
3. If dataset is clean: Proceed with realistic performance expectations


In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv('synthetic_flood_dataset_uganda_seasonal.csv')

print(df.describe())

       MonsoonIntensity  TopographyDrainage  RiverManagement  Deforestation  \
count      10000.000000        10000.000000     10000.000000   10000.000000   
mean           0.500128            0.498461         0.424615       0.333875   
std            0.286962            0.223355         0.193652       0.199332   
min           -0.081760            0.001511         0.004952       0.000874   
25%            0.227714            0.324627         0.273459       0.173745   
50%            0.498525            0.500109         0.422578       0.310650   
75%            0.773318            0.672087         0.567509       0.470672   
max            1.048475            0.991493         0.936734       0.961029   

       Urbanization  ClimateChange   DamsQuality     Siltation  \
count  10000.000000   10000.000000  10000.000000  10000.000000   
mean       0.377320       0.442142      0.668457      0.373772   
std        0.235773       0.211510      0.199293      0.217196   
min        0.000543     